## AI Assignment 2
### Umar Asghar
### 24i - 2532
### BS - DS (B)

In [1]:
import random
import copy

In [2]:
class Card:
    def __init__(self, color, value):
        self.color = color  # Red, Blue, Green, Yellow
        self.value = value  # 0-9 or 'Skip'

    def __repr__(self):
        return f"{self.color} {self.value}"

In [3]:
def create_deck():
    # Making a simple deck with 4 colors and 0-9 + Skips
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    values = list(range(10)) + ['Skip']
    deck = [Card(c, v) for c in colors for v in values]
    random.shuffle(deck)
    return deck


In [4]:
def get_score(state, player_idx, strategy):
    hand = state['hands'][player_idx]
    c_ai = len(hand)

    # Average of what the other two guys have
    others = [i for i in range(3) if i != player_idx]
    c_opp = sum(len(state['hands'][i]) for i in others) / 2

    # Count how many skips we are holding
    s_cards = sum(1 for c in hand if c.value == 'Skip')

    # Tuning weights: Defensive cares about skips, Offensive cares about speed
    if strategy == "Defensive":
        return 50 - 4*(c_ai) + 2*(c_opp) + 6*(s_cards)
    else:
        return 60 - 8*(c_ai) + 3*(c_opp) + 2*(s_cards)

In [5]:
def get_legal_moves(hand, top_card):
    # Rule: Match color or match number
    return [c for c in hand if c.color == top_card.color or c.value == top_card.value]

In [6]:
# Implement minimax and expectimax :)

In [7]:
def minimax(state, depth, p_idx, maximizing):
    # Base case: depth reached or someone won
    if depth == 0 or any(len(h) == 0 for h in state['hands']):
        return get_score(state, 0, "Defensive"), None

    moves = get_legal_moves(state['hands'][p_idx], state['top_card'])
    possible_actions = moves if moves else [None] # None means draw a card

    if maximizing:
        best_val = -float('inf')
        best_move = None
        for move in possible_actions:
            next_s = simulate_move(state, move, p_idx)
            val, _ = minimax(next_s, depth - 1, (p_idx + 1) % 3, False)
            if val > best_val:
                best_val, best_move = val, move
        return best_val, best_move
    else:
        worst_val = float('inf')
        for move in possible_actions:
            next_s = simulate_move(state, move, p_idx)
            val, _ = minimax(next_s, depth - 1, (p_idx + 1) % 3, True)
            if val < worst_val:
                worst_val = val
        return worst_val, None

In [8]:
def simulate_move(state, move, player_idx):
    new_state = copy.deepcopy(state)

    # Check if 'move' is a Card object and not a string like "DRAW_CHANCE"
    if move and not isinstance(move, str):
        # Remove card from hand
        new_state['hands'][player_idx] = [c for c in new_state['hands'][player_idx]
                                         if not (c.color == move.color and c.value == move.value)]
        new_state['top_card'] = move
        if move.value == 'Skip':
            new_state['skip_flag'] = True

    else: # It's a draw action (either None or "DRAW_CHANCE")
        if new_state['deck']:
            new_card = new_state['deck'].pop()
            new_state['hands'][player_idx].append(new_card)

    return new_state

In [9]:
def expectimax(state, depth, p_idx):
    if depth == 0 or any(len(h) == 0 for h in state['hands']):
        return get_score(state, 1, "Offensive"), None

    moves = get_legal_moves(state['hands'][p_idx], state['top_card'])

    # AI Turn (MAX Node)
    if p_idx == 1:
        best_val = -float('inf')
        best_move = None

        # If no moves, we must draw (CHANCE node)
        actions = moves if moves else ["DRAW_CHANCE"]

        for act in actions:
            if act == "DRAW_CHANCE":
                expected_val = 0
                if state['deck']:
                    prob = 1.0 / len(state['deck'])
                    for card in state['deck']:
                        temp_s = copy.deepcopy(state)
                        temp_s['hands'][1].append(card)
                        v, _ = expectimax(temp_s, depth - 1, 2)
                        expected_val += prob * v
                val = expected_val
            else:
                next_s = simulate_move(state, act, 1)
                val, _ = expectimax(next_s, depth - 1, 2)

            if val > best_val:
                best_val, best_move = val, act
        return best_val, best_move

    # Opponent Turn (Random/Average)
    else:
        opp_moves = get_legal_moves(state['hands'][p_idx], state['top_card']) or [None]
        total_val = 0
        for m in opp_moves:
            next_s = simulate_move(state, m, p_idx)
            v, _ = expectimax(next_s, depth - 1, (p_idx + 1) % 3)
            total_val += v
        return total_val / len(opp_moves), None

In [10]:
# 5. The Main Game
def start_simulation():
    deck = create_deck()
    # Deal 5 cards to 3 players
    hands = [[deck.pop() for _ in range(5)] for _ in range(3)]
    top_card = deck.pop()
    game_state = {'hands': hands, 'top_card': top_card, 'deck': deck, 'skip_flag': False}

    current_player = 0
    print(f"Starting Game! Top Card: {top_card}")

    while all(len(h) > 0 for h in game_state['hands']):
        # Show status
        print(f"\n--- Player {current_player}'s Turn ---")
        print(f"Top Card: {game_state['top_card']}")

        if game_state['skip_flag']:
            print(f"Player {current_player} is skipped!")
            game_state['skip_flag'] = False
            current_player = (current_player + 1) % 3
            continue

        chosen_move = None
        if current_player == 0: # P1 - Minimax
            _, chosen_move = minimax(game_state, 3, 0, True)
        elif current_player == 1: # P2 - Expectimax
            _, chosen_move = expectimax(game_state, 3, 1)
        else: # P3 - Simulation Mode (using Minimax)
            _, chosen_move = minimax(game_state, 3, 2, True)

        print(f"Decision: {chosen_move if chosen_move else 'Draw Card'}")
        game_state = simulate_move(game_state, chosen_move, current_player)

        # Check if someone won
        if len(game_state['hands'][current_player]) == 0:
            print(f"\nPlayer {current_player} won the game!")
            break

        current_player = (current_player + 1) % 3

if __name__ == "__main__":
    start_simulation()

Starting Game! Top Card: Green 2

--- Player 0's Turn ---
Top Card: Green 2
Decision: Green 9

--- Player 1's Turn ---
Top Card: Green 9
Decision: DRAW_CHANCE

--- Player 2's Turn ---
Top Card: Green 9
Decision: Draw Card

--- Player 0's Turn ---
Top Card: Green 9
Decision: Green 0

--- Player 1's Turn ---
Top Card: Green 0
Decision: Green 6

--- Player 2's Turn ---
Top Card: Green 6
Decision: Draw Card

--- Player 0's Turn ---
Top Card: Green 6
Decision: Green 8

--- Player 1's Turn ---
Top Card: Green 8
Decision: DRAW_CHANCE

--- Player 2's Turn ---
Top Card: Green 8
Decision: Blue 8

--- Player 0's Turn ---
Top Card: Blue 8
Decision: Blue 5

--- Player 1's Turn ---
Top Card: Blue 5
Decision: Blue 6

--- Player 2's Turn ---
Top Card: Blue 6
Decision: Blue 0

--- Player 0's Turn ---
Top Card: Blue 0
Decision: Blue 9

Player 0 won the game!
